# OASIS Phoenix Notebook
Structure minimale pour reproduire la cellule de visualisation (Cellule 9).

## Cellule 2
Chargement des bibliothèques.

## Cellule 3
Chargement des données.

## Cellule 4
Préparation des séries temporelles.

## Cellule 5
Entraînement des modèles (placeholder).

## Cellule 6
Évaluation des anomalies (placeholder).

## Cellule 7
Préparation des sorties Phoenix (placeholder).

## Cellule 8
Résumé et préparation visualisations.

In [5]:
import os
import sys
import subprocess
from pathlib import Path

for pkg in ('pandas', 'PIL'):
    try:
        __import__(pkg if pkg != 'PIL' else 'PIL.Image')
    except ModuleNotFoundError:
        install_name = 'pillow' if pkg == 'PIL' else pkg
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', install_name])

import pandas as pd
from PIL import Image, ImageDraw

workspace_root = Path.cwd().parent if Path.cwd().name.lower() == 'data' else Path.cwd()
local_data = workspace_root / 'data' / 'combined_clean.csv'
legacy_data = Path(r'C:/Users/Nebrasse/Desktop/Ai-Healing-Gabes/oasis_phoenix/data/combined_clean.csv')
data_path = local_data if local_data.exists() else legacy_data

if not data_path.exists():
    raise FileNotFoundError(f'Dataset introuvable: {data_path}')

os.makedirs('static', exist_ok=True)
df = pd.read_csv(data_path)

so2 = (df['SO2_column_number_density'].fillna(0) * 1e6).astype(float).tolist()
ndvi = df['NDVI'].fillna(0).astype(float).tolist()

w, h = 1400, 900
img = Image.new('RGB', (w, h), '#050d12')
draw = ImageDraw.Draw(img)

m_left, m_right = 80, 40
plot_w = w - m_left - m_right
panel_h = 320
gap = 80
top1 = 70
bot1 = top1 + panel_h
top2 = bot1 + gap
bot2 = top2 + panel_h

for y in (top1, bot1, top2, bot2):
    draw.line([(m_left, y), (w - m_right, y)], fill='#355164', width=1)
draw.line([(m_left, top1), (m_left, bot1)], fill='#355164', width=1)
draw.line([(m_left, top2), (m_left, bot2)], fill='#355164', width=1)

draw.text((m_left, 20), 'SO2 (ug/m3) - Serie temporelle', fill='#ff9fb0')
draw.text((m_left, top2 - 35), 'NDVI - Serie temporelle', fill='#8dffbe')


def draw_series(values, ymin, ymax, y_top, y_bottom, color):
    if not values:
        return
    n = len(values)
    rng = max(1e-9, ymax - ymin)
    pts = []
    for i, v in enumerate(values):
        x = m_left + int((i / max(1, n - 1)) * plot_w)
        y_norm = (v - ymin) / rng
        y = y_bottom - int(y_norm * (y_bottom - y_top))
        pts.append((x, y))
    if len(pts) > 1:
        draw.line(pts, fill=color, width=2)


so2_min, so2_max = min(so2), max(so2)
if so2_min == so2_max:
    so2_max = so2_min + 1.0

draw_series(so2, so2_min, so2_max, top1, bot1, '#ff3355')
draw_series(ndvi, 0.0, 1.0, top2, bot2, '#00ff88')

out_path = workspace_root / 'static' / 'analysis.png'
img.save(out_path)
print(f'Graphe sauvegarde dans {out_path}')

Graphe sauvegarde dans c:\Users\Nebrasse\Desktop\Ai-Healing-Gabes\app\oasis_phoenix\static\analysis.png
